In [112]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================

# 1. LOAD DATA

# =========================

df = pd.read_csv("../data/raw/StudentPerformanceFactors.csv")

print("Original shape:", df.shape)
display(df.head())

# =========================

# 2. DROP MISSING VALUES

# =========================

df = df.dropna()

print("Shape after dropping missing values:", df.shape)

# =========================

# 3. OPTIONAL: CHECK DUPLICATES

# =========================

print("Duplicate rows:", df.duplicated().sum())

# =========================

# 4. SEPARATE FEATURES AND TARGET

# =========================

X = df.drop("Exam_Score", axis=1)

y = df["Exam_Score"]

print("X shape:", X.shape)
print("y shape:", y.shape)

# =========================

# 5. IDENTIFY COLUMN TYPES

# =========================

numerical_columns = X.select_dtypes(
include=["int64", "float64"]
).columns

categorical_columns = X.select_dtypes(
include=["object"]
).columns

print("\nNumerical columns:")
print(list(numerical_columns))

print("\nCategorical columns:")
print(list(categorical_columns))

# =========================

# 6. ONE-HOT ENCODE

# =========================

encoder = OneHotEncoder(
handle_unknown="ignore",
sparse_output=False
)

encoded_data = encoder.fit_transform(
X[categorical_columns]
)

encoded_df = pd.DataFrame(
encoded_data,
columns=encoder.get_feature_names_out(categorical_columns),
index=X.index
)

# Remove original categorical columns

X = X.drop(columns=categorical_columns)

# Add one-hot encoded columns

X = pd.concat([X, encoded_df], axis=1)

print("\nFinal X shape after encoding:", X.shape)

display(X.head())

# =========================

# 7. TRAIN / TEST SPLIT

# =========================

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42
)

print("\nTraining data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

# =========================

# 8. RANDOM FOREST MODEL

# =========================

rf_model = RandomForestRegressor(
n_estimators=300,
random_state=42,
n_jobs=-1
)

rf_model.fit(X_train, y_train)

# =========================

# 9. MAKE PREDICTIONS

# =========================

y_pred = rf_model.predict(X_test)

# =========================

# 10. EVALUATE MODEL

# =========================

mae = mean_absolute_error(
y_test,
y_pred
)

mse = mean_squared_error(
y_test,
y_pred
)

rmse = mse ** 0.5

r2 = r2_score(
y_test,
y_pred
)

print("\n===== RANDOM FOREST RESULTS =====")

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

# =========================

# 11. SEE ACTUAL VS PREDICTED

# =========================

results = pd.DataFrame({
"Actual": y_test,
"Predicted": y_pred,
"Error": y_test - y_pred
})

display(results.head(20))


Original shape: (6607, 20)


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


Shape after dropping missing values: (6378, 20)
Duplicate rows: 0
X shape: (6378, 19)
y shape: (6378,)

Numerical columns:
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity']

Categorical columns:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Motivation_Level', 'Internet_Access', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']

Final X shape after encoding: (6378, 40)


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Parental_Involvement_High,Parental_Involvement_Low,Parental_Involvement_Medium,Access_to_Resources_High,...,Learning_Disabilities_No,Learning_Disabilities_Yes,Parental_Education_Level_College,Parental_Education_Level_High School,Parental_Education_Level_Postgraduate,Distance_from_Home_Far,Distance_from_Home_Moderate,Distance_from_Home_Near,Gender_Female,Gender_Male
0,23,84,7,73,0,3,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
1,19,64,8,59,2,4,0.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,24,98,7,91,2,4,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,29,89,8,98,1,4,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,19,92,6,65,3,4,0.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0



Training data:
(5102, 40)

Testing data:
(1276, 40)

===== RANDOM FOREST RESULTS =====
MAE: 1.126551724137931
RMSE: 2.385427370694158
R² Score: 0.6338152560922159


,Actual,Predicted,Error
2814,74,73.950000,0.050000
4421,66,66.660000,-0.660000
4282,70,69.876667,0.123333
1246,72,71.163333,0.836667
4699,67,66.846667,0.153333
235,69,68.933333,0.066667
1679,72,70.383333,1.616667
300,66,66.623333,-0.623333
5600,69,66.280000,2.720000
5444,68,68.083333,-0.083333
